# 畳み込み層のパラメータと誤差逆伝播法

このノートブックでは、**畳み込み層のパラメータの影響範囲**と**誤差逆伝播法**について学びます。

---

## このノートブックで学ぶこと

1. 全結合層のパラメータによる損失関数の偏微分（復習・完成）
2. **畳み込み層のパラメータの影響範囲**を考察する
3. **畳み込み層のパラメータによる損失関数の偏微分**を導出する
4. **誤差逆伝播法（バックプロパゲーション）**の意味を理解する

---

## 用語の説明

| 用語 | わかりやすい説明 |
|-----|----------------|
| **連鎖律** | 合成関数を微分するときの公式。関数の「鎖」をたどって掛け算する |
| **誤差逆伝播法** | 出力層から入力層へ向かって、誤差を「逆向き」に伝えて勾配を計算する方法 |
| **勾配** | 「どの方向にどれくらい変えれば改善するか」を示す値 |
| **学習率 η** | パラメータを更新する際の「ステップの大きさ」 |

---

## 対応する教科書のセクション
- 3-12: 畳み込み層のパラメータの影響範囲を考察する ー合成関数ー
- 3-13: 畳み込み層のパラメータによる損失関数の偏微分結果を導出する
- 3-14: 誤差逆伝播法の意味を数理モデルから読み取る

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

---

## 1. 前回の復習: 損失関数の偏微分

### 1.1 合成関数の微分（式3-23〜3-25）

前回、ソフトマックス関数 $f(fc) = \frac{1}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$ を合成関数として扱い、

$$f(fc) = (g(fc))^{-1}, \quad g(fc) = e^{fc_0} + e^{fc_1} + e^{fc_2}$$

と置いて連鎖律を適用しました。

### 1.2 計算結果

**式(3-23)**
$$\frac{\partial P_1}{\partial fc_0} = -P_1 P_0$$

**式(3-24)**
$$\frac{\partial P_2}{\partial fc_0} = -P_2 P_0$$

これらの結果を式(3-17)に代入して計算を進めます。

In [ ]:
# ソフトマックスの偏微分の確認

def softmax(fc):
    """ソフトマックス関数"""
    exp_fc = np.exp(fc - np.max(fc))  # 数値安定性のため
    return exp_fc / np.sum(exp_fc)

# 具体的な値で確認
fc = np.array([1.0, 0.5, 0.2])  # fc_0, fc_1, fc_2
P = softmax(fc)

print("=== ソフトマックスの偏微分の確認 ===")
print(f"fc = {fc}")
print(f"P  = [P_0, P_1, P_2] = [{P[0]:.4f}, {P[1]:.4f}, {P[2]:.4f}]")
print()
print("【理論値】式(3-18), (3-23), (3-24)より:")
print(f"  ∂P_0/∂fc_0 = P_0(1-P_0) = {P[0]:.4f} × (1 - {P[0]:.4f}) = {P[0]*(1-P[0]):.4f}")
print(f"  ∂P_1/∂fc_0 = -P_1×P_0   = -{P[1]:.4f} × {P[0]:.4f}    = {-P[1]*P[0]:.4f}")
print(f"  ∂P_2/∂fc_0 = -P_2×P_0   = -{P[2]:.4f} × {P[0]:.4f}    = {-P[2]*P[0]:.4f}")

### 1.3 損失関数の fc_0 に関する偏微分（式3-25）

式(3-17)より:

$$\frac{\partial L(P)}{\partial fc_0} = \frac{\partial L(P)}{\partial P_0} \frac{\partial P_0}{\partial fc_0} + \frac{\partial L(P)}{\partial P_1} \frac{\partial P_1}{\partial fc_0} + \frac{\partial L(P)}{\partial P_2} \frac{\partial P_2}{\partial fc_0}$$

各項を代入:

$$= \frac{-t_0}{P_0} P_0(1-P_0) + \frac{-t_1}{P_1}(-P_1 P_0) + \frac{-t_2}{P_2}(-P_2 P_0)$$

$$= -t_0(1-P_0) + t_1 P_0 + t_2 P_0$$

$$= P_0(t_0 + t_1 + t_2) - t_0$$

正解ラベルは one-hot なので $t_0 + t_1 + t_2 = 1$:

$$\therefore \frac{\partial L(P)}{\partial fc_0} = P_0 - t_0 \quad \text{...(3-25)}$$

**非常にシンプルな結果！「予測 - 正解」という形になりました。**

In [ ]:
# 式(3-25)の計算過程を詳しく確認

print("=== 式(3-25)の導出: ∂L(P)/∂fc_0 ===")
print()

# 正解ラベル（クラス0が正解の場合）
t = np.array([1, 0, 0])

print("【計算の流れ】")
print()
print("∂L(P)/∂fc_0 = ∂L/∂P_0 × ∂P_0/∂fc_0 + ∂L/∂P_1 × ∂P_1/∂fc_0 + ∂L/∂P_2 × ∂P_2/∂fc_0")
print()

# 各項の計算
term1 = (-t[0]/P[0]) * P[0]*(1-P[0])
term2 = (-t[1]/P[1]) * (-P[1]*P[0]) if t[1] != 0 else 0
term3 = (-t[2]/P[2]) * (-P[2]*P[0]) if t[2] != 0 else 0

print(f"第1項: (-t_0/P_0) × P_0(1-P_0) = (-{t[0]}/{P[0]:.4f}) × {P[0]:.4f}×{1-P[0]:.4f}")
print(f"      = -t_0(1-P_0) = -{t[0]}×{1-P[0]:.4f} = {term1:.4f}")
print()
print(f"第2項: (-t_1/P_1) × (-P_1×P_0) = t_1×P_0 = {t[1]}×{P[0]:.4f} = {t[1]*P[0]:.4f}")
print(f"第3項: (-t_2/P_2) × (-P_2×P_0) = t_2×P_0 = {t[2]}×{P[0]:.4f} = {t[2]*P[0]:.4f}")
print()

# 合計
result = -t[0]*(1-P[0]) + t[1]*P[0] + t[2]*P[0]
print(f"合計: -t_0(1-P_0) + t_1×P_0 + t_2×P_0")
print(f"    = P_0(t_0 + t_1 + t_2) - t_0")
print(f"    = P_0 × 1 - t_0  (∵ t_0 + t_1 + t_2 = 1)")
print(f"    = P_0 - t_0")
print(f"    = {P[0]:.4f} - {t[0]} = {P[0] - t[0]:.4f}")
print()
print(f"∴ ∂L(P)/∂fc_0 = P_0 - t_0 = {P[0] - t[0]:.4f}")

### 1.4 全結合層の重みに関する偏微分（式3-26）

式(3-16)と(3-25)より:

$$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = \frac{\partial L(P)}{\partial fc_0} \frac{\partial fc_0}{\partial w_{1,0}^{fc}} = \frac{\partial L(P)}{\partial fc_0} z_1 = (P_0 - t_0) z_1$$

$$\therefore \frac{\partial L(P)}{\partial w_{1,0}^{fc}} = (P_0 - t_0) z_1 \quad \text{...(3-26)}$$

**意味**: $(P_0 - t_0)$ は予測値と正解値の差（誤差）です。

同様に:
- $\frac{\partial L(P)}{\partial fc_1} = P_1 - t_1$ ...(3-25')
- $\frac{\partial L(P)}{\partial fc_2} = P_2 - t_2$

In [ ]:
# 式(3-26)の確認

print("=== 式(3-26): 全結合層の重みに関する偏微分 ===")
print()

z1 = 2.5  # プーリング層の出力
z2 = 1.8

# 勾配の計算
dL_dw10 = (P[0] - t[0]) * z1

print(f"z_1 = {z1}, z_2 = {z2}")
print(f"P_0 = {P[0]:.4f}, t_0 = {t[0]}")
print()
print(f"∂L(P)/∂w_{{1,0}}^{{fc}} = (P_0 - t_0) × z_1")
print(f"                       = ({P[0]:.4f} - {t[0]}) × {z1}")
print(f"                       = {P[0] - t[0]:.4f} × {z1}")
print(f"                       = {dL_dw10:.4f}")
print()
print("【解釈】")
print(f"  誤差 (P_0 - t_0) = {P[0] - t[0]:.4f} は負の値")
print(f"  → P_0 が t_0 より小さい = 正解クラスの確率が低すぎる")
print(f"  → 重みを増やして P_0 を大きくする必要がある")

---

## 2. 畳み込み層のパラメータの影響範囲（3-12節）

### 2.1 図3.30: CNNの構造と影響範囲

次は、畳み込み層の重み $w_{1,1}^1$ を最適化することを考えます。

図3.30のように、$w_{1,1}^1$ は $c_{1,1}^1$ と $z_1$ を経由して出力に影響を与えています。
つまり、$w_{1,0}^{fc}$ よりも**影響範囲が広い**ということです。

In [ ]:
# 図3.30: CNNの構造図

fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)

ax.text(9, 9.5, '図3.30: 更新の対象となる畳み込み層のパラメータ及びその影響を受ける関数', 
        fontsize=13, fontweight='bold', ha='center')

# 層のヘッダー
headers = [
    (1.5, '入力層'),
    (4, '畳み込み層\n(ストライド幅:1)'),
    (7, 'プーリング層'),
    (10.5, '全結合層'),
    (13.5, '出力層'),
    (16, '正解値')
]
for x, label in headers:
    ax.text(x, 8.5, label, fontsize=10, ha='center', fontweight='bold')

# 入力層 (3x3)
for i in range(3):
    for j in range(3):
        rect = Rectangle((0.5 + j*0.6, 7 - i*0.6), 0.5, 0.5, 
                         facecolor='lightblue', edgecolor='black')
        ax.add_patch(rect)
        ax.text(0.75 + j*0.6, 7.25 - i*0.6, f'$x_{{{i+1},{j+1}}}$', 
                fontsize=7, ha='center', va='center')

# 畳み込み層（フィルタ）
for k in range(2):  # 2つのフィルタ
    y_offset = 6.5 - k*2.5
    for i in range(2):
        for j in range(2):
            color = 'yellow' if k == 0 and i == 0 and j == 0 else 'lightyellow'
            rect = Rectangle((3.2 + j*0.5, y_offset - i*0.5), 0.45, 0.45,
                             facecolor=color, edgecolor='black', linewidth=1.5 if color=='yellow' else 1)
            ax.add_patch(rect)
            ax.text(3.42 + j*0.5, y_offset - i*0.5 + 0.22, f'$w_{{{i+1},{j+1}}}^{k+1}$',
                    fontsize=6, ha='center', va='center')
    
    # 特徴マップ
    for i in range(2):
        for j in range(2):
            rect = Rectangle((4.5 + j*0.5, y_offset - i*0.5), 0.45, 0.45,
                             facecolor='lightgreen', edgecolor='black')
            ax.add_patch(rect)
            ax.text(4.72 + j*0.5, y_offset - i*0.5 + 0.22, f'$c_{{{i+1},{j+1}}}^{k+1}$',
                    fontsize=6, ha='center', va='center')

# 黄色でハイライト: w_{1,1}^1
ax.annotate('$w_{1,1}^1$\n(この重みを\n最適化)', xy=(3.45, 6.7), xytext=(2.2, 7.5),
            fontsize=9, ha='center', color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

# プーリング層
for i, y in enumerate([6.5, 4.5]):
    rect = Rectangle((6.5, y), 1, 0.8, facecolor='lightgray', edgecolor='black')
    ax.add_patch(rect)
    ax.text(7, y + 0.4, f'$z_{i+1}$', fontsize=10, ha='center', va='center')

# 全結合層
fc_positions = [(10.5, 7), (10.5, 5.5), (10.5, 4)]
fc_labels = ['$fc_0$', '$fc_1$', '$fc_2$']
for (x, y), label in zip(fc_positions, fc_labels):
    circle = plt.Circle((x, y), 0.35, facecolor='orange', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=9, ha='center', va='center')

# 出力層
out_positions = [(13.5, 7), (13.5, 5.5), (13.5, 4)]
out_labels = ['$P_0$', '$P_1$', '$P_2$']
for (x, y), label in zip(out_positions, out_labels):
    circle = plt.Circle((x, y), 0.35, facecolor='coral', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=9, ha='center', va='center')

# 正解値
t_positions = [(16, 7), (16, 5.5), (16, 4)]
t_labels = ['$t_0$', '$t_1$', '$t_2$']
for (x, y), label in zip(t_positions, t_labels):
    circle = plt.Circle((x, y), 0.35, facecolor='lightgreen', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=9, ha='center', va='center')

# プーリング→全結合の接続（重み）
weights_fc = [
    ((7.5, 6.9), (10.15, 7), '$w_{1,0}^{fc}$'),
    ((7.5, 6.5), (10.15, 5.5), '$w_{1,1}^{fc}$'),
    ((7.5, 4.9), (10.15, 7), '$w_{2,0}^{fc}$'),
]
for (x1, y1), (x2, y2), label in weights_fc:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

# 全結合→出力の接続
for fx, fy in fc_positions:
    for ox, oy in out_positions:
        ax.plot([fx + 0.35, ox - 0.35], [fy, oy], 'gray', alpha=0.3, lw=0.5)

# 出力→正解の接続
for (ox, oy), (tx, ty) in zip(out_positions, t_positions):
    ax.annotate('', xy=(tx - 0.35, ty), xytext=(ox + 0.35, oy),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

# 影響範囲のハイライト
highlight = FancyBboxPatch((5.8, 3.3), 8.5, 4.5, boxstyle='round,pad=0.1',
                            facecolor='none', edgecolor='red', linestyle='--', linewidth=2)
ax.add_patch(highlight)

ax.text(10, 2.5, '$w_{1,1}^1$ を変えると、赤い点線の中すべてに影響！', 
        fontsize=11, ha='center', color='red')
ax.text(10, 1.8, '($z_1$ → $fc_0, fc_1, fc_2$ → $P_0, P_1, P_2$ → $L$)', 
        fontsize=10, ha='center', color='red')

ax.axis('off')
plt.tight_layout()
plt.show()

### 2.2 各関数の定義（図3.30より）

損失関数は先程と同じ $L(P)$ です。$P_0, fc_0$ も同様です。

**式(3-12)** 損失関数:
$$L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$$

**式(3-13)** ソフトマックス:
$$P_0 = s_0(fc_0, fc_1, fc_2) = \frac{e^{fc_0(z_1, z_2)}}{\sum_{k=0}^{2} e^{fc_k(z_1, z_2)}}$$

**式(3-14)** 全結合層:
$$fc_0 = fc_0(z_1, z_2) = w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2$$

ここで、$c_{1,1}^1$ も式(3-27)のように関数とみなすことができます。$f$ は ReLU 関数です。

**式(3-27)**
$$c_{1,1}^1 = f(w_{1,1}^1 x_{1,1} + w_{1,2}^1 x_{1,2} + w_{2,1}^1 x_{2,1} + w_{2,2}^1 x_{2,2})$$

### 2.3 z_1 の定義（式3-28, 3-29）

$c_{1,1}^1$ については、ReLU関数の処理の結果、

**式(3-28)**
$$c_{1,1}^1 = w_{1,1}^1 x_{1,1} + w_{1,2}^1 x_{1,2} + w_{2,1}^1 x_{2,1} + w_{2,2}^1 x_{2,2}$$

だとします。さらに、Max Pooling の処理によって $z_1$ は $c_{1,1}^1, c_{1,2}^1, c_{2,1}^1, c_{2,2}^1$ の中で最大のものとイコールです。

ここでは $z_1 = c_{1,1}^1$ だとして考察を進めます。つまり、

**式(3-29)**
$$z_1 = c_{1,1}^1 = w_{1,1}^1 x_{1,1} + w_{1,2}^1 x_{1,2} + w_{2,1}^1 x_{2,1} + w_{2,2}^1 x_{2,2}$$

$z_1$ は式(3-29)の通り入力層の値と重みパラメータによる関数と見ることができます。

In [ ]:
# 式(3-27)〜(3-29)の数値例

print("=== z_1 の計算例 ===")
print()

# 入力画像の一部
x = np.array([[0.5, 0.8, 0.3],
              [0.2, 0.9, 0.1],
              [0.4, 0.6, 0.7]])

# 畳み込みフィルタの重み
w1 = np.array([[0.3, 0.5],
               [0.2, 0.4]])

print("入力画像 x:")
print(x)
print()
print("畳み込みフィルタ w^1:")
print(w1)
print()

# 畳み込み計算
c11 = w1[0,0]*x[0,0] + w1[0,1]*x[0,1] + w1[1,0]*x[1,0] + w1[1,1]*x[1,1]

print("c_{1,1}^1 = w_{1,1}^1 × x_{1,1} + w_{1,2}^1 × x_{1,2} + w_{2,1}^1 × x_{2,1} + w_{2,2}^1 × x_{2,2}")
print(f"         = {w1[0,0]} × {x[0,0]} + {w1[0,1]} × {x[0,1]} + {w1[1,0]} × {x[1,0]} + {w1[1,1]} × {x[1,1]}")
print(f"         = {w1[0,0]*x[0,0]:.2f} + {w1[0,1]*x[0,1]:.2f} + {w1[1,0]*x[1,0]:.2f} + {w1[1,1]*x[1,1]:.2f}")
print(f"         = {c11:.2f}")
print()
print(f"Max Pooling で z_1 = c_{{1,1}}^1 = {c11:.2f} とすると")
print(f"z_1 は入力 x と重み w^1 の関数として表される")

---

## 3. 畳み込み層のパラメータによる損失関数の偏微分（3-13節）

### 3.1 連鎖律の適用（式3-30）

以上の前提のもと、$w_{1,1}^1$ の最適化を考えます。連鎖律によって式(3-30)の数式を作ることができます。

$$\frac{\partial L(P)}{\partial w_{1,1}^1} = \frac{\partial L(P)}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_{1,1}^1} \quad \text{...(3-30)}$$

### 3.2 ∂z_1/∂w_{1,1}^1 の計算（式3-31）

まず、$\frac{\partial z_1}{\partial w_{1,1}^1}$ は簡単に計算できます。$w_{1,1}^1$ に関係のない項が消えていることを確認してください。

$$\frac{\partial z_1}{\partial w_{1,1}^1} = \frac{\partial}{\partial w_{1,1}^1}(w_{1,1}^1 x_{1,1} + w_{1,2}^1 x_{1,2} + w_{2,1}^1 x_{2,1} + w_{2,2}^1 x_{2,2}) = x_{1,1} \quad \text{...(3-31)}$$

In [ ]:
# 式(3-31)の確認

print("=== 式(3-31): ∂z_1/∂w_{1,1}^1 の計算 ===")
print()
print("z_1 = w_{1,1}^1 × x_{1,1} + w_{1,2}^1 × x_{1,2} + w_{2,1}^1 × x_{2,1} + w_{2,2}^1 × x_{2,2}")
print()
print("w_{1,1}^1 で偏微分すると:")
print("  ∂(w_{1,1}^1 × x_{1,1})/∂w_{1,1}^1 = x_{1,1}  ← w_{1,1}^1 を含む項")
print("  ∂(w_{1,2}^1 × x_{1,2})/∂w_{1,1}^1 = 0       ← w_{1,1}^1 を含まない")
print("  ∂(w_{2,1}^1 × x_{2,1})/∂w_{1,1}^1 = 0       ← w_{1,1}^1 を含まない")
print("  ∂(w_{2,2}^1 × x_{2,2})/∂w_{1,1}^1 = 0       ← w_{1,1}^1 を含まない")
print()
print("∴ ∂z_1/∂w_{1,1}^1 = x_{1,1}")
print()
print(f"具体的な値: x_{{1,1}} = {x[0,0]}")

### 3.3 ∂L(P)/∂z_1 の計算（式3-32, 3-33）

次に、$\frac{\partial L(P)}{\partial z_1}$ について考えます。

図3.30の通り、$z_1$ は $fc_0, fc_1, fc_2$ のすべてに影響を与えています。また、$fc_0, fc_1, fc_2$ はそれぞれ独立した関数です。

よって、それぞれの偏微分結果を**加算**します。

**式(3-32)**
$$\frac{\partial L(P)}{\partial z_1} = \frac{\partial L(P)}{\partial fc_0} \frac{\partial fc_0}{\partial z_1} + \frac{\partial L(P)}{\partial fc_1} \frac{\partial fc_1}{\partial z_1} + \frac{\partial L(P)}{\partial fc_2} \frac{\partial fc_2}{\partial z_1}$$

式変形によって $\frac{\partial L(P)}{\partial fc_0}, \frac{\partial L(P)}{\partial fc_1}, \frac{\partial L(P)}{\partial fc_2}$ が現れました。

これは先程 $w_{1,0}^{fc}$ の考察で既に計算しましたね。計算結果の式(3-25), 式(3-25')より、それぞれ $(P_0-t_0), (P_1-t_1), (P_2-t_2)$ となるのでした。

さらに、式(3-14)より
$$\frac{\partial fc_0}{\partial z_1} = \frac{\partial}{\partial z_1}(w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2) = w_{1,0}^{fc}$$

同様に計算すると $\frac{\partial fc_1}{\partial z_1} = w_{1,1}^{fc}$, $\frac{\partial fc_2}{\partial z_1} = w_{1,2}^{fc}$ となります。

これらを式(3-32)に代入しましょう。

**式(3-33)**
$$\frac{\partial L(P)}{\partial z_1} = (P_0 - t_0) w_{1,0}^{fc} + (P_1 - t_1) w_{1,1}^{fc} + (P_2 - t_2) w_{1,2}^{fc}$$

In [ ]:
# 式(3-32), (3-33)の計算

print("=== 式(3-32), (3-33): ∂L(P)/∂z_1 の計算 ===")
print()

# 全結合層の重み
w_fc = np.array([[0.5, 0.3, 0.2],   # z_1 から fc_0, fc_1, fc_2 への重み
                 [0.4, 0.5, 0.1]])  # z_2 から fc_0, fc_1, fc_2 への重み

print("全結合層の重み:")
print(f"  w_{{1,0}}^{{fc}} = {w_fc[0,0]} (z_1 → fc_0)")
print(f"  w_{{1,1}}^{{fc}} = {w_fc[0,1]} (z_1 → fc_1)")
print(f"  w_{{1,2}}^{{fc}} = {w_fc[0,2]} (z_1 → fc_2)")
print()

# 予測と正解の差（誤差）
errors = P - t
print("予測と正解の差（誤差）:")
print(f"  P_0 - t_0 = {P[0]:.4f} - {t[0]} = {errors[0]:.4f}")
print(f"  P_1 - t_1 = {P[1]:.4f} - {t[1]} = {errors[1]:.4f}")
print(f"  P_2 - t_2 = {P[2]:.4f} - {t[2]} = {errors[2]:.4f}")
print()

# ∂L(P)/∂z_1 の計算
dL_dz1 = errors[0]*w_fc[0,0] + errors[1]*w_fc[0,1] + errors[2]*w_fc[0,2]

print("式(3-33):")
print("∂L(P)/∂z_1 = (P_0-t_0)×w_{1,0}^{fc} + (P_1-t_1)×w_{1,1}^{fc} + (P_2-t_2)×w_{1,2}^{fc}")
print(f"           = {errors[0]:.4f}×{w_fc[0,0]} + {errors[1]:.4f}×{w_fc[0,1]} + {errors[2]:.4f}×{w_fc[0,2]}")
print(f"           = {errors[0]*w_fc[0,0]:.4f} + {errors[1]*w_fc[0,1]:.4f} + {errors[2]*w_fc[0,2]:.4f}")
print(f"           = {dL_dz1:.4f}")

### 3.4 最終結果

式(3-31), (3-33)を式(3-30)に代入することで次の式が得られます。

$$\frac{\partial L(P)}{\partial w_{1,1}^1} = \frac{\partial L(P)}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_{1,1}^1}$$

$$= \{(P_0 - t_0) w_{1,0}^{fc} + (P_1 - t_1) w_{1,1}^{fc} + (P_2 - t_2) w_{1,2}^{fc}\} x_{1,1}$$

In [ ]:
# 畳み込み層の重みに関する勾配

print("=== 畳み込み層の重みに関する勾配 ===")
print()

# ∂L(P)/∂w_{1,1}^1
x11 = x[0, 0]
dL_dw11 = dL_dz1 * x11

print("∂L(P)/∂w_{1,1}^1 = ∂L(P)/∂z_1 × ∂z_1/∂w_{1,1}^1")
print(f"                = {dL_dz1:.4f} × {x11}")
print(f"                = {dL_dw11:.4f}")
print()
print("展開すると:")
print("∂L(P)/∂w_{1,1}^1 = {(P_0-t_0)w_{1,0}^{fc} + (P_1-t_1)w_{1,1}^{fc} + (P_2-t_2)w_{1,2}^{fc}} × x_{1,1}")

---

## 4. パラメータの影響範囲の比較（図3.31）

これで、与えられた条件下での「全結合層のパラメータ」及び「畳み込み層のパラメータ」を偏微分計算するための数理モデルの導出に成功しました。

それぞれのパラメータの位置付けを図3.31で再度確認しましょう。「木を見て森を見ず」にならないことが大切です。

In [ ]:
# 図3.31: パラメータごとの影響範囲の比較

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

for ax_idx, (ax, title, highlight_fc, highlight_conv) in enumerate([
    (axes[0], '全結合層のパラメータ最適化', True, False),
    (axes[1], '畳み込み層のパラメータ最適化', False, True)
]):
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # 入力層
    ax.text(1.5, 7.2, '入力層', fontsize=9, ha='center', fontweight='bold')
    for i in range(3):
        for j in range(3):
            rect = Rectangle((0.5 + j*0.6, 5.5 - i*0.6), 0.5, 0.5,
                             facecolor='lightblue', edgecolor='black')
            ax.add_patch(rect)
    
    # 畳み込み層
    ax.text(4, 7.2, '畳み込み層', fontsize=9, ha='center', fontweight='bold')
    conv_color = 'yellow' if highlight_conv else 'lightyellow'
    for i in range(2):
        for j in range(2):
            color = conv_color if i == 0 and j == 0 else 'lightyellow'
            rect = Rectangle((3.2 + j*0.5, 5.5 - i*0.5), 0.45, 0.45,
                             facecolor=color, edgecolor='black',
                             linewidth=2 if color == 'yellow' else 1)
            ax.add_patch(rect)
    
    # 特徴マップ
    for i in range(2):
        for j in range(2):
            rect = Rectangle((4.5 + j*0.5, 5.5 - i*0.5), 0.45, 0.45,
                             facecolor='lightgreen', edgecolor='black')
            ax.add_patch(rect)
    
    # プーリング層
    ax.text(6.5, 7.2, 'プーリング層', fontsize=9, ha='center', fontweight='bold')
    for i, y in enumerate([5.5, 4.5]):
        rect = Rectangle((6, y), 1, 0.8, facecolor='lightgray', edgecolor='black')
        ax.add_patch(rect)
        ax.text(6.5, y + 0.4, f'$z_{i+1}$', fontsize=10, ha='center', va='center')
    
    # 全結合層
    ax.text(9.5, 7.2, '全結合層', fontsize=9, ha='center', fontweight='bold')
    fc_color = 'yellow' if highlight_fc else 'orange'
    fc_positions = [(9.5, 6), (9.5, 5), (9.5, 4)]
    for idx, (fx, fy) in enumerate(fc_positions):
        color = fc_color if idx == 0 else 'orange'
        circle = plt.Circle((fx, fy), 0.35, facecolor=color, edgecolor='black',
                           linewidth=2 if color == 'yellow' else 1)
        ax.add_patch(circle)
        ax.text(fx, fy, f'$fc_{idx}$', fontsize=9, ha='center', va='center')
    
    # 重みの線（ハイライト）
    if highlight_fc:
        ax.annotate('', xy=(9.15, 6), xytext=(7, 5.9),
                    arrowprops=dict(arrowstyle='->', color='red', lw=3))
        ax.text(8, 6.5, '$w_{1,0}^{fc}$', fontsize=10, color='red', fontweight='bold')
    
    # 出力層
    ax.text(12, 7.2, '出力層', fontsize=9, ha='center', fontweight='bold')
    out_positions = [(12, 6), (12, 5), (12, 4)]
    for idx, (ox, oy) in enumerate(out_positions):
        circle = plt.Circle((ox, oy), 0.35, facecolor='coral', edgecolor='black')
        ax.add_patch(circle)
        ax.text(ox, oy, f'$P_{idx}$', fontsize=9, ha='center', va='center')
    
    # 正解値
    ax.text(14.5, 7.2, '正解値', fontsize=9, ha='center', fontweight='bold')
    t_positions = [(14.5, 6), (14.5, 5), (14.5, 4)]
    for idx, (tx, ty) in enumerate(t_positions):
        circle = plt.Circle((tx, ty), 0.35, facecolor='lightgreen', edgecolor='black')
        ax.add_patch(circle)
        ax.text(tx, ty, f'$t_{idx}$', fontsize=9, ha='center', va='center')
    
    # 接続線
    for fx, fy in fc_positions:
        for ox, oy in out_positions:
            ax.plot([fx + 0.35, ox - 0.35], [fy, oy], 'gray', alpha=0.3, lw=0.5)
    
    # 影響範囲のハイライト
    if highlight_fc:
        highlight = FancyBboxPatch((8.8, 3.3), 6.5, 3.5, boxstyle='round,pad=0.1',
                                    facecolor='none', edgecolor='blue', linestyle='--', linewidth=2)
        ax.add_patch(highlight)
        ax.text(12, 2.5, '影響範囲: $fc_0$ → $P_0, P_1, P_2$ → $L$', fontsize=10, color='blue', ha='center')
    else:
        highlight = FancyBboxPatch((5.5, 3.3), 9.8, 3.5, boxstyle='round,pad=0.1',
                                    facecolor='none', edgecolor='red', linestyle='--', linewidth=2)
        ax.add_patch(highlight)
        ax.text(10, 2.5, '影響範囲: $z_1$ → $fc_0, fc_1, fc_2$ → $P_0, P_1, P_2$ → $L$', 
                fontsize=10, color='red', ha='center')
    
    ax.axis('off')

plt.tight_layout()
plt.show()

print("【比較】")
print("・全結合層のパラメータ: fc_0 以降に影響（範囲が狭い）")
print("・畳み込み層のパラメータ: z_1 以降すべてに影響（範囲が広い）")
print()
print("畳み込み層のパラメータは、より多くの関数を通過するため、")
print("連鎖律でより長い「鎖」をたどる必要があります。")

---

## 5. 誤差逆伝播法の意味を数理モデルから読み取る（3-14節）

### 5.1 導出した数式のまとめ

さて、ここで導出した数式を並べてみましょう。図3.31と突き合わせて確認してください。

**全結合層の重みに関する勾配:**
$$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = (P_0 - t_0) z_1$$

**畳み込み層の重みに関する勾配:**
$$\frac{\partial L(P)}{\partial w_{1,1}^1} = \{(P_0 - t_0) w_{1,0}^{fc} + (P_1 - t_1) w_{1,1}^{fc} + (P_2 - t_2) w_{1,2}^{fc}\} x_{1,1}$$

### 5.2 勾配降下法による更新式

これらを用いて勾配降下法による**更新式**を次のように設計できます。設計の仕方は第2章で解説した通りです。

$$w_{1,0}^{fc(1)} = w_{1,0}^{fc(0)} - \eta \cdot \frac{\partial L(P)}{\partial w_{1,0}^{fc(0)}}$$

$$w_{1,1}^{1(1)} = w_{1,1}^{1(0)} - \eta \cdot \frac{\partial L(P)}{\partial w_{1,1}^{1(0)}}$$

$\eta$（イータ）は**学習率**を表しますが、これも第2章で解説しました。$w$ の右上に表記されている $(0)$ は更新が0回目、つまり未更新のパラメータだということを示しています。

In [ ]:
# 更新式の実装

print("=== 勾配降下法による重みの更新 ===")
print()

eta = 0.1  # 学習率

# 現在の重み
w_fc_10_old = w_fc[0, 0]
w_conv_11_old = w1[0, 0]

print(f"学習率 η = {eta}")
print()
print("【全結合層の重み更新】")
print(f"  更新前: w_{{1,0}}^{{fc(0)}} = {w_fc_10_old}")
print(f"  勾配:   ∂L/∂w_{{1,0}}^{{fc}} = (P_0 - t_0) × z_1 = {dL_dw10:.4f}")

w_fc_10_new = w_fc_10_old - eta * dL_dw10
print(f"  更新後: w_{{1,0}}^{{fc(1)}} = {w_fc_10_old} - {eta} × {dL_dw10:.4f}")
print(f"                         = {w_fc_10_new:.4f}")
print()

print("【畳み込み層の重み更新】")
print(f"  更新前: w_{{1,1}}^{{1(0)}} = {w_conv_11_old}")
print(f"  勾配:   ∂L/∂w_{{1,1}}^1 = {dL_dw11:.4f}")

w_conv_11_new = w_conv_11_old - eta * dL_dw11
print(f"  更新後: w_{{1,1}}^{{1(1)}} = {w_conv_11_old} - {eta} × {dL_dw11:.4f}")
print(f"                        = {w_conv_11_new:.4f}")

### 5.3 誤差逆伝播法とは

ここまでの計算過程を振り返ると、重要なパターンが見えてきます。

1. まず**出力層**で「予測 - 正解」という**誤差**を計算
2. その誤差を**逆方向**（出力→入力）に伝播させながら各層の勾配を計算
3. 計算した勾配を使って重みを更新

この方法を**誤差逆伝播法（バックプロパゲーション）**と呼びます。

**「逆伝播」の意味:**
- 順伝播: 入力 → 畳み込み → プーリング → 全結合 → 出力（前向き）
- 逆伝播: 出力 → 全結合 → プーリング → 畳み込み → 入力（後ろ向き）

誤差が「逆向き」に伝わっていくイメージです。

In [ ]:
# 誤差逆伝播法の図解

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)

ax.text(8, 7.5, '誤差逆伝播法（バックプロパゲーション）のイメージ', 
        fontsize=14, fontweight='bold', ha='center')

# 層のボックス
layers = [
    (2, '入力層', 'lightblue'),
    (5, '畳み込み層', 'lightyellow'),
    (8, 'プーリング層', 'lightgray'),
    (11, '全結合層', 'orange'),
    (14, '出力層', 'coral'),
]

for x, label, color in layers:
    rect = FancyBboxPatch((x - 1.2, 4), 2.4, 1.5, boxstyle='round,pad=0.1',
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 4.75, label, fontsize=10, ha='center', va='center', fontweight='bold')

# 順伝播の矢印（上）
ax.text(8, 6.5, '順伝播（Forward Propagation）', fontsize=12, ha='center', color='green')
for i in range(len(layers) - 1):
    ax.annotate('', xy=(layers[i+1][0] - 1.3, 5.7), xytext=(layers[i][0] + 1.3, 5.7),
                arrowprops=dict(arrowstyle='->', color='green', lw=2))

# 逆伝播の矢印（下）
ax.text(8, 2.5, '逆伝播（Backpropagation）', fontsize=12, ha='center', color='red')
for i in range(len(layers) - 1, 0, -1):
    ax.annotate('', xy=(layers[i-1][0] + 1.3, 3.3), xytext=(layers[i][0] - 1.3, 3.3),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))

# 誤差の伝播
ax.text(14, 3, '誤差\n$P-t$', fontsize=10, ha='center', color='red', fontweight='bold')
ax.text(11, 3, '勾配\n計算', fontsize=9, ha='center', color='red')
ax.text(8, 3, '勾配\n計算', fontsize=9, ha='center', color='red')
ax.text(5, 3, '勾配\n計算', fontsize=9, ha='center', color='red')

# 説明
ax.text(8, 1.2, '連鎖律を使って、誤差を「後ろから前へ」伝播させながら各層の勾配を計算', 
        fontsize=11, ha='center')
ax.text(8, 0.5, '計算した勾配で重みを更新 → これを繰り返してネットワークを学習', 
        fontsize=11, ha='center')

ax.axis('off')
plt.tight_layout()
plt.show()

### 5.4 誤差逆伝播法の効率性

誤差逆伝播法が優れている点は、**計算の効率性**にあります。

例えば、畳み込み層の勾配を計算するとき:

$$\frac{\partial L(P)}{\partial w_{1,1}^1} = \{(P_0 - t_0) w_{1,0}^{fc} + (P_1 - t_1) w_{1,1}^{fc} + (P_2 - t_2) w_{1,2}^{fc}\} x_{1,1}$$

中括弧の中身は $\frac{\partial L(P)}{\partial z_1}$ であり、これは全結合層の勾配計算で**既に計算済み**です。

つまり、前の層の計算結果を**再利用**できるのです。これにより、各層の勾配を効率的に計算できます。

In [ ]:
# 計算の再利用の確認

print("=== 誤差逆伝播法の効率性 ===")
print()
print("【ステップ1】出力層での誤差計算")
print(f"  P_0 - t_0 = {errors[0]:.4f}")
print(f"  P_1 - t_1 = {errors[1]:.4f}")
print(f"  P_2 - t_2 = {errors[2]:.4f}")
print()

print("【ステップ2】全結合層の勾配")
print(f"  ∂L/∂fc_0 = P_0 - t_0 = {errors[0]:.4f}  ← 誤差をそのまま使用")
print(f"  ∂L/∂w_{{1,0}}^{{fc}} = (P_0 - t_0) × z_1 = {dL_dw10:.4f}")
print()

print("【ステップ3】プーリング層への逆伝播")
print(f"  ∂L/∂z_1 = Σ(誤差 × 重み) = {dL_dz1:.4f}  ← ステップ1の結果を再利用")
print()

print("【ステップ4】畳み込み層の勾配")
print(f"  ∂L/∂w_{{1,1}}^1 = ∂L/∂z_1 × x_{{1,1}} = {dL_dw11:.4f}  ← ステップ3の結果を再利用")
print()
print("→ 各ステップで前のステップの計算結果を再利用！")
print("→ これが誤差逆伝播法の効率性の秘密")

---

## 6. 完全な実装例

最後に、ここまで学んだ内容を使って、簡単なニューラルネットワークの学習を実装してみましょう。

In [ ]:
class SimpleNetwork:
    """シンプルな2層ニューラルネットワーク"""
    
    def __init__(self):
        # 重みの初期化
        np.random.seed(42)
        self.w_conv = np.random.randn(2, 2) * 0.5  # 畳み込み層
        self.w_fc = np.random.randn(2, 3) * 0.5    # 全結合層
    
    def forward(self, x):
        """順伝播"""
        # 畳み込み（簡略化: 1点のみ）
        self.x = x[:2, :2]  # 2x2の入力
        self.c = np.sum(self.w_conv * self.x)  # 畳み込み出力
        
        # ReLU
        self.z = max(0, self.c)
        
        # 全結合層（z1のみ使用、z2=0と仮定）
        self.fc = self.z * self.w_fc[0, :]  # [fc0, fc1, fc2]
        
        # ソフトマックス
        exp_fc = np.exp(self.fc - np.max(self.fc))
        self.P = exp_fc / np.sum(exp_fc)
        
        return self.P
    
    def backward(self, t):
        """逆伝播"""
        # 出力層の誤差
        self.error = self.P - t  # (P - t)
        
        # 全結合層の勾配
        self.dw_fc = np.outer([self.z, 0], self.error)
        
        # プーリング層への逆伝播
        self.dz = np.sum(self.error * self.w_fc[0, :])
        
        # ReLUの逆伝播
        self.dc = self.dz if self.c > 0 else 0
        
        # 畳み込み層の勾配
        self.dw_conv = self.dc * self.x
        
        return self.dw_conv, self.dw_fc
    
    def update(self, lr=0.1):
        """重みの更新"""
        self.w_conv -= lr * self.dw_conv
        self.w_fc -= lr * self.dw_fc
    
    def loss(self, t):
        """クロスエントロピー損失"""
        return -np.sum(t * np.log(self.P + 1e-10))

# 学習の実行
net = SimpleNetwork()

# 入力と正解
x = np.array([[0.5, 0.8, 0.3],
              [0.2, 0.9, 0.1],
              [0.4, 0.6, 0.7]])
t = np.array([1, 0, 0])  # クラス0が正解

losses = []
probs = []

print("=== 学習の進行 ===")
print()

for epoch in range(100):
    # 順伝播
    P = net.forward(x)
    L = net.loss(t)
    
    losses.append(L)
    probs.append(P.copy())
    
    # 逆伝播
    net.backward(t)
    
    # 重み更新
    net.update(lr=0.5)
    
    if epoch < 5 or (epoch + 1) % 20 == 0:
        print(f"エポック{epoch+1:3d}: 損失={L:.4f}, P_0={P[0]:.4f}")

print()
print(f"最終的な正解クラスの確率: {probs[-1][0]*100:.1f}%")

In [ ]:
# 学習の可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 損失の推移
ax = axes[0]
ax.plot(losses, 'b-', linewidth=2)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('損失関数の推移', fontsize=14)
ax.grid(True, alpha=0.3)

# 確率の推移
ax = axes[1]
probs_array = np.array(probs)
ax.plot(probs_array[:, 0], 'r-', linewidth=2, label='$P_0$（正解クラス）')
ax.plot(probs_array[:, 1], 'g--', linewidth=2, label='$P_1$')
ax.plot(probs_array[:, 2], 'b:', linewidth=2, label='$P_2$')
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('確率', fontsize=12)
ax.set_title('各クラスの予測確率の推移', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

print("誤差逆伝播法によって、正解クラスの確率が上昇しています！")

---

## まとめ

### 今日学んだこと

| 概念 | 説明 |
|-----|------|
| **式(3-25)** | $\frac{\partial L}{\partial fc_0} = P_0 - t_0$（予測 - 正解）|
| **式(3-26)** | $\frac{\partial L}{\partial w_{1,0}^{fc}} = (P_0 - t_0) z_1$ |
| **式(3-33)** | $\frac{\partial L}{\partial z_1} = \sum_k (P_k - t_k) w_{1,k}^{fc}$ |
| **畳み込み層の勾配** | $\frac{\partial L}{\partial w_{1,1}^1} = \frac{\partial L}{\partial z_1} \cdot x_{1,1}$ |
| **誤差逆伝播法** | 出力層から入力層へ誤差を逆向きに伝播させて勾配を計算 |

### 重要なポイント

1. **連鎖律**により、複雑なネットワークでも勾配を計算できる
2. **誤差逆伝播法**は計算結果を再利用するため効率的
3. 畳み込み層のパラメータは全結合層より**影響範囲が広い**
4. 最終的な勾配は「**誤差 × 入力**」という直感的な形になる

### 次のステップ

これで、CNNの数学的な基礎は完成です。次は実際のデータ（MNIST等）を使った学習に進みましょう。